# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [ ]:
import pandas as pd

url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

df = df.drop(columns=["Unnamed: 0"])              # leftover index column from however this was exported
df.columns = df.columns.str.lower().str.replace(" ", "_")
df["effective_to_date"] = pd.to_datetime(df["effective_to_date"], format="%m/%d/%y")
df["month"] = df["effective_to_date"].dt.month     # needed for tasks 5-6

# 1. low total_claim_amount AND responded "Yes"
low_claim_responders = df[(df["total_claim_amount"] < 1000) & (df["response"] == "Yes")]
print("1.", low_claim_responders.shape)
low_claim_responders.head()

# 2. monthly premium / CLV by policy_type and gender, responders only -- vs total_claim_amount
yes_df = df[df["response"] == "Yes"]

premium_clv = yes_df.groupby(["policy_type", "gender"])[["monthly_premium_auto", "customer_lifetime_value"]].mean().round(2)
print("\n2a. premium & CLV:\n", premium_clv)

claims = yes_df.groupby(["policy_type", "gender"])["total_claim_amount"].mean().round(2)
print("\n2b. total_claim_amount:\n", claims)
# Personal Auto (F) stands out: highest CLV (~8340) among these groups, but a mid-pack claim
# amount (~453) -- more revenue-per-customer without a matching rise in payouts, which is
# the profile of a low-risk, profitable segment. Special Auto (M) is close behind on CLV
# (~8247) with the lowest claim amount of any group (~430) -- worth a similar look.
# By contrast, Personal Auto (M) has the lowest CLV here (~7448) but one of the highest
# claim amounts (~457) -- paying out relatively more per customer while bringing in less.

# 3. customers per state, filtered to > 500
customers_by_state = df.groupby("state")["customer"].count()
states_over_500 = customers_by_state[customers_by_state > 500]
print("\n3.\n", states_over_500)

# 4. CLV max/min/median by education and gender
clv_by_edu_gender = df.groupby(["education", "gender"])["customer_lifetime_value"].agg(["max", "min", "median"]).round(2)
print("\n4.\n", clv_by_edu_gender)
# The max CLV varies a lot by group (from ~32.7k for Doctor/M up to ~83.3k for High School
# or Below/M), but the median is remarkably flat across every education/gender combination
# (roughly 5.3k-6.3k everywhere) -- education level doesn't move the *typical* customer's
# value much, it just changes how far the high-end outliers reach.

## Bonus

# 5. policies sold by state and month, months as columns
policies_by_state_month = df.pivot_table(
    index="state", columns="month", values="number_of_policies", aggfunc="sum"
)
print("\n5.\n", policies_by_state_month)

# 6. top 3 states by total policies sold, policies by month for those 3 only
top3_states = df.groupby("state")["number_of_policies"].sum().sort_values(ascending=False).head(3).index
print("\ntop 3 states:", list(top3_states))

top3_by_month = policies_by_state_month.loc[top3_states]
print("\n6.\n", top3_by_month)

# 7. response rate by marketing channel -- build the wide count table, then melt it back
# into a clean long-format (channel, response, count) table
counts_wide = df.pivot_table(index="sales_channel", columns="response", values="customer", aggfunc="count")
counts_wide["response_rate_pct"] = (counts_wide["Yes"] / (counts_wide["Yes"] + counts_wide["No"]) * 100).round(1)
print("\n7. counts + rate:\n", counts_wide)

response_long = counts_wide[["No", "Yes"]].reset_index().melt(
    id_vars="sales_channel", var_name="response", value_name="count"
)
print("\nmelted long format:\n", response_long)
# Agent has both the most customers reached AND the best response rate (19.1%) -- roughly
# 70% higher than any other channel (11-12% for Branch/Call Center/Web). Worth investigating
# what Agent is doing differently before assuming it's just a bigger audience.